In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from openbabel import openbabel, pybel
import os
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import shutil

In [ ]:

df = pd.read_excel('Dimer.xlsx')
dimer_excel_path = 'Dimer.xlsx'

In [ ]:

nproc = 10
mem = 20

In [ ]:

def calculate_charge(smiles):
    
    mol = Chem.MolFromSmiles(smiles)  
    if mol is None:  
        return None
    mol = Chem.AddHs(mol)  
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())  
    return charge

In [ ]:

def get_filename_without_extension(file_path):
    
    
    base_name = os.path.basename(file_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def normalization(mol):
    smi=Chem.MolToSmiles(mol)
    n_mol=Chem.MolFromSmiles(smi)
    return n_mol

In [ ]:

def normalization_SMILES(df, SMILESname="SMILES"):
    
    if f"{SMILESname}" in df.columns:
        
        invalid_rows = []
        
        for index, row in df.iterrows():
            try:
                
                mol = Chem.MolFromSmiles(row[SMILESname])
                if mol:  
                    
                    n_mol = normalization(mol)
                    if n_mol:  
                        
                        n_smi = Chem.MolToSmiles(n_mol)
                        
                        df.at[index, SMILESname] = n_smi
                    else:
                        
                        invalid_rows.append(index)
                else:
                    
                    invalid_rows.append(index)
            except Exception as e:
                
                print(f"An error occurred for index {index}: {e}")
                invalid_rows.append(index)
        
        
        df.drop(invalid_rows, inplace=True)
        return df
        
    else:
        print(f"The {SMILESname} column does not exist in the provided Excel file.")

In [ ]:

def clean_name_column(df, name_col):
    

    
    illegal_chars = [",", ".", "/", "\\", " "]

    
    
    pattern = '|'.join([f"\\{char}" for char in illegal_chars])

    
    
    df[f'{name_col}'] = df[f'{name_col}'].str.replace(pattern, "-", regex=True)

    
    return df

In [ ]:

def drop_duplicates(df,SMILESname="SMILES"):
    
    
    df.drop_duplicates(subset=SMILESname, keep='first', inplace=True)
    
    print("Public message removed for release.")

In [ ]:

df = clean_name_column(df, name_col="Dimer Name")
df = clean_name_column(df, name_col="Component Name A")
df = clean_name_column(df, name_col="Component Name B")


df = normalization_SMILES(df, SMILESname="Component SMILES A")
df = normalization_SMILES(df, SMILESname="Component SMILES B")
df = normalization_SMILES(df, SMILESname="Dimer SMILES")


drop_duplicates(df, SMILESname="Dimer SMILES")


df.to_excel('Dimer.xlsx', index=False)

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:

def generate_lowest_energy_conformer_openbabel(smiles: str, num_confs: int = 50, forcefield: str = "UFF"):
    
    
    obConversion = openbabel.OBConversion()
    obConversion.SetInFormat("smi")
    
    
    obmol = openbabel.OBMol()
    obConversion.ReadString(obmol, smiles)
    
    
    obmol.AddHydrogens()
    
    
    builder = openbabel.OBBuilder()
    builder.Build(obmol)
    
    
    cs = openbabel.OBConformerSearch()
    cs.Setup(obmol, num_confs, True)
    cs.Search()
    cs.GetConformers(obmol)
    
    
    forcefield = forcefield.upper()
    if forcefield == "MMFF94":
        ff = openbabel.OBForceField.FindForceField("mmff94")
    elif forcefield == "UFF":
        ff = openbabel.OBForceField.FindForceField("uff")
    else:
        raise ValueError("Public message removed for release.")
    
    lowest_energy = float('inf')
    lowest_conf_index = None
    nconfs = obmol.NumConformers()
    
    
    for i in range(nconfs):
        obmol.SetConformer(i)
        if not ff.Setup(obmol):
            print("Public message removed for release.")
            continue
        ff.ConjugateGradients(250, 1.0e-4)
        ff.GetCoordinates(obmol)
        energy = ff.Energy()
        if energy < lowest_energy:
            lowest_energy = energy
            lowest_conf_index = i

    
    if lowest_conf_index is None:
        return None, None, None
    
    atomic_number_to_symbol = {
        1: "H",    2: "He",   3: "Li",   4: "Be",   5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",
        11: "Na",  12: "Mg",  13: "Al",  14: "Si",  15: "P",   16: "S",   17: "Cl",  18: "Ar",  19: "K",   20: "Ca",
        21: "Sc",  22: "Ti",  23: "V",   24: "Cr",  25: "Mn",  26: "Fe",  27: "Co",  28: "Ni",  29: "Cu",  30: "Zn",
        31: "Ga",  32: "Ge",  33: "As",  34: "Se",  35: "Br",  36: "Kr",  37: "Rb",  38: "Sr",  39: "Y",   40: "Zr",
        41: "Nb",  42: "Mo",  43: "Tc",  44: "Ru",  45: "Rh",  46: "Pd",  47: "Ag",  48: "Cd",  49: "In",  50: "Sn",
        51: "Sb",  52: "Te",  53: "I",   54: "Xe",  55: "Cs",  56: "Ba",  57: "La",  58: "Ce",  59: "Pr",  60: "Nd",
        61: "Pm",  62: "Sm",  63: "Eu",  64: "Gd",  65: "Tb",  66: "Dy",  67: "Ho",  68: "Er",  69: "Tm",  70: "Yb",
        71: "Lu",  72: "Hf",  73: "Ta",  74: "W",   75: "Re",  76: "Os",  77: "Ir",  78: "Pt",  79: "Au",  80: "Hg",
        81: "Tl",  82: "Pb",  83: "Bi",  84: "Po",  85: "At",  86: "Rn",  87: "Fr",  88: "Ra",  89: "Ac",  90: "Th",
        91: "Pa",  92: "U",   93: "Np",  94: "Pu",  95: "Am",  96: "Cm",  97: "Bk",  98: "Cf",  99: "Es", 100: "Fm",
        101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg", 107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds",
        111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl", 115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }

    
    obmol.SetConformer(lowest_conf_index)
    coordinates = []
    
    for atom in pybel.ob.OBMolAtomIter(obmol):
        atomic_num = atom.GetAtomicNum()
        symbol = atomic_number_to_symbol.get(atomic_num, str(atomic_num))  
        x = atom.GetX()
        y = atom.GetY()
        z = atom.GetZ()
        coordinates.append((symbol, x, y, z))
    
    return lowest_conf_index, lowest_energy, coordinates

In [ ]:
def create_Gaussian_inputfile(name, smiles):

    
    lowest_conf_index, lowest_energy, coordinates_list = generate_lowest_energy_conformer_openbabel(smiles, num_confs=50, forcefield="UFF")

    
    if lowest_conf_index is None:
        print("Public message removed for release.")
        return
    
    
    coordinate_str = ""
    for atom_type, x, y, z in coordinates_list:
        coordinate_str += f"{atom_type} {x:.6f} {y:.6f} {z:.6f}\n"

    
    filename = name.replace(' ', '_') + '.gjf'

    
    
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, filename.replace('.gjf', '.chk'))

    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles)

    
    
    with open(filename, 'w') as f:
        f.write(f"%nprocshared={nproc}\n")
        f.write(f"%mem={mem}GB\n")
        f.write(f"%chk={chk_file}\n")
        f.write("# opt freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682\n\n")
        f.write(f"{filename} opt+freq gas\n\n")
        f.write(f"{charge} {multiplicity}\n")
        f.write(coordinate_str)
        
        
        f.write("\n\n")

In [ ]:

def create_or_load_molecule_xlsx(directory):
    
    xlsx_path = os.path.join(directory, 'molecule.xlsx')
    
    
    if not os.path.exists(xlsx_path):
        
        df = pd.DataFrame(columns=['FileName', 'SMILES'])
        
        df.to_excel(xlsx_path, index=False)
    else:
        
        df = pd.read_excel(xlsx_path)
    
    return df

In [ ]:

def normalization(mol):
    smi=Chem.MolToSmiles(mol)
    n_mol=Chem.MolFromSmiles(smi)
    return n_mol

In [ ]:

def normalization_SMILES(excel_path, SMILESName="SMILES"):
    
    df = pd.read_excel(excel_path)
    file_name_without_extension = get_filename_without_extension(excel_path)
    
    
    if SMILESName in df.columns:
        
        for index, row in df.iterrows():
            
            mol = Chem.MolFromSmiles(row[SMILESName])
            if mol is not None:  
                
                n_mol = normalization(mol)
                
                n_smi = Chem.MolToSmiles(n_mol)
                
                df.at[index, SMILESName] = n_smi

        
        df.to_excel(f'{file_name_without_extension}.xlsx', index=False)
    else:
        print(f"The {SMILESName} column does not exist in the provided Excel file.")

In [ ]:

def read_excel_to_dict(excel_path, key_col, value_col):
    
    try:
        
        df = pd.read_excel(excel_path)
        
        
        if key_col not in df.columns or value_col not in df.columns:
            raise ValueError("Public message removed for release.")
        
        
        result_dict = pd.Series(df[value_col].values,index=df[key_col]).to_dict()
        
        return result_dict
    except FileNotFoundError:
        print("Public message removed for release.")
    except ValueError as e:
        print(e)
    except Exception as e:
        print("Public message removed for release.")

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:
result

In [ ]:
import os  


def check_and_create_gaussian_database(base_dir: str = result['gaussian_database_path']):  
    
    home_directory = os.path.abspath(os.path.expanduser(base_dir))  
    gaussian_database_path = os.path.join(home_directory, 'Gaussian_database')  
    optfreq_gaussian_database_path = os.path.join(gaussian_database_path, 'opt+freq')  
    RESPpolymer_database_path = os.path.join(gaussian_database_path, 'RESPpolymer')  

    if not os.path.exists(gaussian_database_path):  
        os.makedirs(gaussian_database_path)  
        os.makedirs(optfreq_gaussian_database_path)  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(optfreq_gaussian_database_path) and os.path.exists(RESPpolymer_database_path):  
        os.makedirs(optfreq_gaussian_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_gaussian_database_path):  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_gaussian_database_path):  
        os.makedirs(optfreq_gaussian_database_path)  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    else:  
        print("Public message removed for release.")  

    return gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path  

In [ ]:


def compare_smiles_dicts(system_dict, database_dict):
    
    not_found_molecule_dict = {}
    found_molecule_dict = {}

    
    for smiles, name in system_dict.items():
        if smiles not in database_dict.keys():
            not_found_molecule_dict[smiles] = name

    
    for smiles in database_dict:
        if smiles in system_dict:
            name = system_dict[smiles]  
            found_molecule_dict[smiles] = name

    return not_found_molecule_dict, found_molecule_dict

In [ ]:

def read_system_excel_to_dict(excel_path, SMILESName="SMILES", Name="Name"):
    
    return read_excel_to_dict(excel_path, SMILESName, Name)


def read_database_excel_to_dict(database_path):
    
    return read_excel_to_dict(database_path, 'SMILES', 'FileName')

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:
def copy_files_based_on_smiles(database_directory, database_excel_path, found_molecule_dict, destination_directory):
    
    
    if not os.path.exists(destination_directory):
        raise Exception(f"{destination_directory} does not exist")

    
    df = pd.read_excel(database_excel_path)

    
    for smiles, name in found_molecule_dict.items():
        
        matched_files = df[df['SMILES'] == smiles]['FileName'].tolist()
        
        
        for matched_file in matched_files:
            
            for ext in ['.gjf', '.chk', '.out']:
                file_to_copy = matched_file + ext
                source_path = os.path.join(database_directory, file_to_copy)
                
                if os.path.isfile(source_path):
                    
                    new_name = str(name) + ext
                    destination_path = os.path.join(destination_directory, new_name)
                    
                    shutil.copy2(source_path, destination_path)
                    print("Public message removed for release.")
                else:
                    print("Public message removed for release.")

In [ ]:
from qc_database_utils import (
    add_and_normalize_smiles,
    compare_smiles_dicts,
    copy_files_based_on_smiles,
    ensure_database_excel,
    get_gaussian_database_paths,
    read_database_excel_to_dict,
)

if 'result' not in globals():
    from cemp_software_settings import load_and_apply_settings
    result = load_and_apply_settings()


In [ ]:



gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path = get_gaussian_database_paths(result)


molecule_database_path = optfreq_gaussian_database_path
polymer_database_path = RESPpolymer_database_path


molecule_database_excel_path = ensure_database_excel(molecule_database_path)
polymer_database_excel_path = ensure_database_excel(polymer_database_path)


molecule_database = read_database_excel_to_dict(molecule_database_excel_path)  
polymer_database = read_database_excel_to_dict(polymer_database_excel_path)  


In [ ]:

normalization_SMILES(dimer_excel_path, SMILESName="Dimer SMILES")
normalization_SMILES(dimer_excel_path, SMILESName="Component SMILES A")
normalization_SMILES(dimer_excel_path, SMILESName="Component SMILES B")


componentA_system_dict = read_system_excel_to_dict(dimer_excel_path, SMILESName="Component SMILES A", Name="Component Name A")
componentB_system_dict = read_system_excel_to_dict(dimer_excel_path, SMILESName="Component SMILES B", Name="Component Name B")


componentA_not_found_dict, componentA_found_dict = compare_smiles_dicts(componentA_system_dict, molecule_database)
componentB_not_found_dict, componentB_found_dict = compare_smiles_dicts(componentB_system_dict, molecule_database)

In [ ]:

os.makedirs('component_gas/opt+freq/success', exist_ok=True)
os.makedirs('component_gas/opt+freq/failure', exist_ok=True)

os.makedirs("component_gas/opt+freq/imaginary_frequency", exist_ok=True)

base_dir = 'component_gas/opt+freq'
success_dir = os.path.join(base_dir, 'success') 
failure_dir = os.path.join(base_dir, 'failure') 
imaginary_frequency_dir = os.path.join(base_dir, "imaginary_frequency") 

In [ ]:

copy_files_based_on_smiles(molecule_database_path, molecule_database_excel_path, componentA_found_dict, success_dir)
copy_files_based_on_smiles(molecule_database_path, molecule_database_excel_path, componentB_found_dict, success_dir)

In [ ]:


def create_input_files_for_missing_molecules(not_found_molecule_dict):
    
    
    for smiles, name in not_found_molecule_dict.items():
        
        
        create_Gaussian_inputfile(name, smiles)

In [ ]:
create_input_files_for_missing_molecules(componentA_not_found_dict)

In [ ]:
create_input_files_for_missing_molecules(componentB_not_found_dict)